# RAG: 검색증강생성 완전 가이드 - 실습 코드 2: 순수 Python으로 RAG 시스템 구현 (라이브러리 없이)

- Tutorial ID: `expand-rag-fundamentals`
- Tutorial: RAG: 검색증강생성 완전 가이드
- Section ID: `expand-rag-fundamentals-code-2`
- Section: 실습 코드 2: 순수 Python으로 RAG 시스템 구현 (라이브러리 없이)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: 순수 Python으로 RAG 시스템 구현 (라이브러리 없이)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 질문 → 검색 → 근거 선택 → 답변/검증 파이프라인을 단계별로 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
from typing import List, Dict, Tuple
import json

# ── 1. 간단한 임베딩 (TF-IDF 스타일) ──
class SimpleEmbedder:
    """TF-IDF 기반 간단한 임베딩 (데모용)"""
        def __init__(self, max_features=5000):
        self.vocab = {}
        self.idf = {}
        self.max_features = max_features
    
        def fit(self, documents: List[str]):
        """어휘 및 IDF 계산"""
        # 단어 빈도수 계산
        doc_freq = {}
        total_docs = len(documents)
                for doc in documents:
            words = set(doc.lower().split())
                        for word in words:
                doc_freq[word] = doc_freq.get(word, 0) + 1
        
        # 상위 max_features 단어 선택
        sorted_words = sorted(doc_freq.items(), key=lambda x: -x[1])[:self.max_features]
        self.vocab = {word: i for i, (word, _) in enumerate(sorted_words)}
        
        # IDF 계산
        self.idf = {word: np.log(total_docs / (freq + 1)) for word, freq in doc_freq.items()}
        return self
    
        def embed(self, text: str) -> np.ndarray:
        """텍스트 → TF-IDF 벡터"""
        vec = np.zeros(len(self.vocab))
        words = text.lower().split()
        word_count = {}
                for w in words:
            word_count[w] = word_count.get(w, 0) + 1
        
                for word, count in word_count.items():
            if word in self.vocab:
                tf = count / len(words)
                vec[self.vocab[word]] = tf * self.idf.get(word, 1.0)
        
        # L2 정규화
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec


# ── 2. 벡터 저장소 ──
class VectorStore:
    """간단한 벡터 저장소 (코사인 유사도 검색)"""
        def __init__(self):
        self.embeddings = []
        self.documents = []
    
        def add(self, docs: List[Dict], embedder: SimpleEmbedder):
        """문서 추가"""
                for doc in docs:
            emb = embedder.embed(doc['content'])
            self.embeddings.append(emb)
            self.documents.append(doc)
        self.embeddings = np.array(self.embeddings)
    
        def search(self, query_emb: np.ndarray, top_k: int = 3) -> List[Tuple[Dict, float]]:
        """코사인 유사도로 검색"""
        similarities = np.dot(self.embeddings, query_emb)
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return [(self.documents[i], similarities[i]) for i in top_indices]


# ── 3. 텍스트 청커 ──
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """텍스트를 겹치는 청크로 분할"""
    words = text.split()
    chunks = []
        for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks


# ── 4. 완전한 RAG 시스템 ──
class SimpleRAG:
    """라이브러리 없이 구현한 RAG 시스템"""
        def __init__(self, chunk_size=200, chunk_overlap=50):
        self.embedder = SimpleEmbedder()
        self.vectorstore = VectorStore()
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
        def index_documents(self, documents: List[Dict]):
        """문서 인덱싱"""
        # 청킹
        all_chunks = []
                for doc in documents:
            chunks = chunk_text(doc['content'], self.chunk_size, self.chunk_overlap)
                        for i, chunk in enumerate(chunks):
                all_chunks.append({
                    'content': chunk,
                    'source': doc.get('id', 'unknown'),
                    'chunk_id': i,
                    'metadata': doc.get('metadata', {})
                })
        
        # 임베딩 학습 + 인덱싱
        self.embedder.fit([c['content'] for c in all_chunks])
        self.vectorstore.add(all_chunks, self.embedder)
        print(f"✅ {len(all_chunks)}개 청크 인덱싱 완료")
    
        def retrieve(self, query: str, top_k: int = 3) -> List[Tuple[Dict, float]]:
        """관련 문서 검색"""
        query_emb = self.embedder.embed(query)
        return self.vectorstore.search(query_emb, top_k)
    
        def generate_answer(self, query: str, context_docs: List[Tuple[Dict, float]]) -> str:
        """컨텍스트 기반 응답 생성 (간소화 — 실제로는 LLM 사용)"""
        context = "\n".join([doc['content'] for doc, score in context_docs])
        
        # 실제 프로덕션에서는 LLM API 호출
        # 여기서는 검색 결과를 요약하는 간단한 로직 사용
        answer = f"질문: {query}\n\n"
        answer += f"🔍 검색된 관련 문서 ({len(context_docs)}개):\n\n"
                for i, (doc, score) in enumerate(context_docs):
            answer += f"[문서 {i+1}] (유사도: {score:.3f})\n"
            answer += f"출처: {doc['source']}\n"
            answer += f"내용: {doc['content'][:200]}...\n\n"
        
        return answer
    
        def query(self, question: str, top_k: int = 3) -> str:
        """전체 RAG 파이프라인: 검색 → 증강 → 생성"""
        # Step 1: Retrieve
        context_docs = self.retrieve(question, top_k)
        
        # Step 2: Augment + Generate
        answer = self.generate_answer(question, context_docs)
        
        return answer


# ── 실행 예시 ──
knowledge_base = [
    {"id": "transformer_paper", "content": 
     "The Transformer architecture was introduced in the paper Attention Is All You Need by Vaswani et al. in 2017. It replaced recurrent neural networks with self-attention mechanisms, enabling parallel processing of sequences. The model consists of an encoder and decoder, each with multi-head attention and feed-forward networks."},
    {"id": "bert_paper", "content": 
     "BERT introduced masked language modeling for bidirectional pre-training. It achieves state-of-the-art results on 11 NLP benchmarks. BERT-Base has 110M parameters with 12 layers and 768 hidden dimensions."},
    {"id": "gpt_paper", "content": 
     "GPT-3 demonstrated that large language models can perform few-shot learning without fine-tuning. With 175 billion parameters trained on 45TB of data, GPT-3 showed emergent abilities including translation, question answering, and code generation."},
    {"id": "lora_paper", "content": 
     "LoRA enables efficient fine-tuning by decomposing weight updates into low-rank matrices. With only 0.1% of parameters, LoRA achieves comparable performance to full fine-tuning on many tasks."},
]

rag = SimpleRAG()
rag.index_documents(knowledge_base)

# 질의
questions = [
    "What is the Transformer architecture?",
    "How does LoRA reduce fine-tuning cost?",
    "What is few-shot learning?",
]

print("\n=== RAG 질의 응답 ===\n")
for q in questions:
    print(rag.query(q))
    print("=" * 60)